# 03 — Multi-Object Tracking with ByteTrack

YOLO gives us detections per frame — but it has no memory. Player #7 in frame 1 and player #7 in frame 2 might be completely different people in YOLO's output.  

**Tracking** solves this: it assigns a stable ID to each object across frames.

ByteTrack's key idea: keep two pools of tracks — *confirmed* (high confidence) and *tentative* (low confidence). When a detection disappears, keep the tentative track alive for a few frames, then try to re-associate it. This handles occlusions gracefully.

**Connection to the pipeline:** `tracker.get_object_tracks()` in `football_analysis/trackers/tracker.py`

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
from ultralytics import YOLO

from football_analysis.utils.video_utils import read_video

In [ ]:
MODEL_PATH = '../models/best.pt'
VIDEO_PATH = '../input_videos/08fd33_4.mp4'

model   = YOLO(MODEL_PATH)
tracker = sv.ByteTrack()

# Load a short segment (30 frames) to keep this notebook fast
cap = cv2.VideoCapture(VIDEO_PATH)
frames = []
for _ in range(30):
    ret, f = cap.read()
    if ret:
        frames.append(f)
cap.release()
print(f'Loaded {len(frames)} frames')

## Detection vs tracking

First let's see what raw YOLO gives us — no stable IDs.

In [ ]:
results = model.predict(frames[:5], conf=0.1, verbose=False)

for i, r in enumerate(results):
    cls_names = [r.names[int(c)] for c in r.boxes.cls.tolist()]
    print(f'Frame {i}: {len(r.boxes)} detections — {cls_names[:5]}...')

## Wrapping detections for ByteTrack

`supervision.Detections.from_ultralytics()` converts YOLO output into a format ByteTrack understands.  
`tracker.update_with_detections()` returns the same detections but with an added `tracker_id` field.

In [ ]:
results = model.predict(frames, conf=0.1, verbose=False)

all_tracks = []   # list of dicts per frame: {track_id: bbox}

for frame_num, result in enumerate(results):
    cls_names     = result.names
    cls_names_inv = {v: k for k, v in cls_names.items()}

    det_sv = sv.Detections.from_ultralytics(result)

    # Remap goalkeeper → player so ByteTrack tracks them together
    for idx, cls_id in enumerate(det_sv.class_id):
        if cls_names[cls_id] == 'goalkeeper':
            det_sv.class_id[idx] = cls_names_inv['player']

    det_with_tracks = tracker.update_with_detections(det_sv)

    frame_dict = {}
    for det in det_with_tracks:
        bbox     = det[0].tolist()
        cls_id   = det[3]
        track_id = det[4]
        if cls_names[cls_id] == 'player':
            frame_dict[track_id] = bbox

    all_tracks.append(frame_dict)

print(f'Frame 0 player track IDs: {sorted(all_tracks[0].keys())}')
print(f'Frame 1 player track IDs: {sorted(all_tracks[1].keys())}')

## Visualising stable IDs across frames

Each player keeps the same ID across consecutive frames.  
IDs that disappear (player leaves frame) and reappear get re-associated if within the re-identification window.

In [ ]:
def draw_tracks(frame, track_dict, color=(0, 255, 255)):
    """Draw ellipse + ID label for each tracked player."""
    out = frame.copy()
    for tid, bbox in track_dict.items():
        x1, y1, x2, y2 = [int(v) for v in bbox]
        cx = (x1 + x2) // 2
        w  = x2 - x1
        cv2.ellipse(out, (cx, y2), (w//2, int(w*0.35)), 0, -45, 235, color, 2)
        cv2.putText(out, str(tid), (cx - 10, y2 + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 2)
    return out

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, frame_idx in zip(axes, [0, 10, 29]):
    annotated = draw_tracks(frames[frame_idx], all_tracks[frame_idx])
    ax.imshow(annotated[:, :, ::-1])
    ax.set_title(f'Frame {frame_idx} — {len(all_tracks[frame_idx])} players tracked')
    ax.axis('off')

plt.suptitle('ByteTrack — stable player IDs across frames', fontsize=13)
plt.tight_layout()
plt.show()

## Track continuity

How many frames does each player appear in across our 30-frame window?

In [ ]:
from collections import Counter

id_counts = Counter()
for frame_dict in all_tracks:
    id_counts.update(frame_dict.keys())

print('Track ID → frames present:')
for tid, count in sorted(id_counts.items()):
    bar = '█' * count
    print(f'  ID {tid:3d}: {bar} ({count}/30)')

## Ball handling

The ball is tiny and frequently occluded — ByteTrack struggles to give it a stable ID.  
Our pipeline takes a simpler approach: **always store the ball under key `1`**, and handle gaps with pandas interpolation (`interpolate_ball_positions`).

In [ ]:
# Count frames where ball is detected
ball_detected = 0
for result in results:
    for cls_id in result.boxes.cls.tolist():
        if result.names[int(cls_id)] == 'ball':
            ball_detected += 1
            break

print(f'Ball detected in {ball_detected}/{len(frames)} frames ({ball_detected/len(frames)*100:.0f}%)')
print('Missing frames are filled in by linear interpolation.')

## Key takeaways

| Concept | Detail |
|---|---|
| ByteTrack | Associates detections across frames using IoU + Kalman filter prediction |
| Goalkeeper → player | Reclassified before tracking so they get player-style IDs |
| Ball | Stored under a fixed key; gaps filled by interpolation |
| `track_id` | Persists as long as the player stays (or briefly leaves) the frame |

**Next:** `04_team_assignment.ipynb` — using jersey colours to label each player's team